# 5 — BankNifty Enhanced Charts V3 (Split Figures + ROC Average Line)

**Run frequency:** During market hours alongside Notebook 2

**What this does:**
1. Logs in to Zerodha KiteConnect (needed for live ATM strike)
2. Starts a SparkSession
3. Reads instruments + expiries from Parquet
4. Builds the options DataFrame for the configured ATM strike
5. Renders **enhanced V3** Plotly charts:

   | Figure | Layout | Content |
   |--------|--------|---------|
   | 1 | 1×2 | **Straddle Price & Signals** \| **ROC AVG + ROC Average Line** |
   | 2 | 1×2 | **CE vs PE Close** \| **CE/PE ROC & Ratio** *(straddle only)* |
   | 3 | 1×2 | Combined OI \| CE/PE OI + PUT−CALL OI secondary Y |
   | 4 | standalone | PCR |
   | 5 | 2×1 shared-x | Straddle Price/VWAP/OIWAP + PUT−CALL OI indicator pane |
   | 6 | 2×1 shared-x | OHLC Candlestick + Volume |

   All charts use `DD-Mon HH:MM` time labels with vertical day-boundary lines.

6. **Auto-refreshes** every minute; stops automatically at `END_HOUR:END_MINUTE`

---
### ⚙️ Configuration

| Parameter | Default | Description |
|-----------|---------|-------------|
| `STRIKE_LEVEL_NAME` | `'ATM'` | Which strike to chart |
| `CE_OR_PE` | `'E'` | `'CE'`, `'PE'`, or `'E'` for straddle |
| `NUM_DAYS` | `2` | Days of data to display |
| `IS_LATEST_DAY` | `True` | Show only today's data |
| `CUSTOM_STRIKE` | `0` | Override ATM (0 = live LTP) |
| `NUM_STRIKES` | `11` | Strikes each side of ATM to load |
| `LOOP_INTERVAL_MIN` | `1` | Chart refresh frequency (minutes) |
| `END_HOUR` | `15` | Stop loop at this hour (24h, set `None` to run indefinitely) |
| `END_MINUTE` | `30` | Stop loop at this minute |

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
# os.chdir('..')  # Run from repo root so DataFiles/ paths resolve correctly

In [ ]:
# ── CONFIG — edit these values ─────────────────────────────────────────────
STRIKE_LEVEL_NAME  = 'ATM'   # 'ATM', 'ATM+1', 'ATM-2', etc.
CE_OR_PE           = 'E'     # 'CE', 'PE', or 'E' for straddle
NUM_DAYS           = 2       # Days of OHLC data to show
IS_LATEST_DAY      = True    # True = show today only; False = show NUM_DAYS
NUM_DAYS_BACK      = 0       # 0 = latest, 1 = go back 1 day
CUSTOM_STRIKE      = 54500   # 0 = use live LTP; override e.g. 56500
NUM_STRIKES        = 11      # Strikes each side of ATM to load
LOOP_INTERVAL_MIN  = 10      # Chart refresh frequency (minutes)
INTERVAL           = '3minute'

# Time-based loop exit — set to None to run indefinitely
END_HOUR           = 15      # 24-hour format (e.g. 15 = 3 PM)
END_MINUTE         = 30      # Minute (e.g. 30 = :30)

In [ ]:
# ── Step 1: Login ──────────────────────────────────────────────────────────
from utils.kite_helpers import kite_login, get_spark_session

kite, kws, access_token = kite_login()
print('✅ Login successful')

In [ ]:
# ── Step 2: Start SparkSession ─────────────────────────────────────────────
spark = get_spark_session(app_name='5_Enhanced_Charts_BNF')
print('✅ Spark session ready')

In [ ]:
# ── Step 3: Load instruments and expiry data ───────────────────────────────
from utils.strike_utils import read_instruments, get_expiry_dates, get_Options_DF, get_ATM_Strike
from utils.strike_utils import BANKNIFTY_INDEX_TOKEN

bnf_options, bnf_futures, nifty_options, nifty_futures, expiries_df = read_instruments(spark)
bnf_expiries = get_expiry_dates(expiries_df, 'BANKNIFTY')

print(f"BankNifty current week expiry: {bnf_expiries['current_week']}")
print(f"BankNifty next week expiry:    {bnf_expiries['next_week']}")

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*fork.*")
warnings.filterwarnings("ignore", message=".*To enable non-built-in garbage collector.*")
warnings.filterwarnings("ignore", message=".*DeprecationWarning.*")

In [ ]:
# ── Step 4: Build options DataFrame for ATM region ────────────────────────
if CUSTOM_STRIKE > 0:
    bnf_atm = CUSTOM_STRIKE
else:
    bnf_atm = get_ATM_Strike(kite, BANKNIFTY_INDEX_TOKEN, rounding_number=100)

print(f'BankNifty ATM Strike = {bnf_atm}')

Banknifty_Options_DF = get_Options_DF(
    spark                = spark,
    options_df_from_file = bnf_options,
    atm_strike           = bnf_atm,
    current_expiry       = bnf_expiries['current_week'],
    strike_range         = 100,
    num_strikes          = NUM_STRIKES,
)
print(f"Options loaded: {Banknifty_Options_DF.count()} rows")
Banknifty_Options_DF.show()
# display(Banknifty_Options_DF)

In [ ]:
# ── Step 5: Start auto-refreshing ENHANCED V3 charts ──────────────────────
# This cell blocks until END_HOUR:END_MINUTE or Kernel → Interrupt.
#
# Charts rendered each refresh:
#   Fig 1 — Price & Signals  (1×2: Straddle+BB+SL | ROC AVG + ROC Average Line)
#   Fig 2 — CE / PE panels   (1×2: CE/PE Close | CE/PE ROC+Ratio)  [straddle only]
#   Fig 3 — Open Interest    (1×2: Combined OI | CE/PE OI + PUT−CALL OI secondary Y)
#   Fig 4 — PCR              (standalone)
#   Fig 5 — Straddle Price / VWAP / OIWAP  +  PUT−CALL OI indicator pane
#   Fig 6 — OHLC Candlestick + Volume
from utils.enhanced_charts_v3 import run_enhanced_chart_loop_v3

run_enhanced_chart_loop_v3(
    spark                 = spark,
    options_df            = Banknifty_Options_DF,
    expiry                = bnf_expiries['current_week'],
    strike_level_name     = STRIKE_LEVEL_NAME,
    ce_or_pe              = CE_OR_PE,
    interval              = INTERVAL,
    num_days              = NUM_DAYS,
    is_latest_day         = IS_LATEST_DAY,
    loop_interval_minutes = LOOP_INTERVAL_MIN,
    end_hour              = END_HOUR,
    end_minute            = END_MINUTE,
)
